In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

RANDOM_STATE = 42

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

modeling_data = pd.read_csv(
    DATA_INTERIM / "modeling_data.csv",
    dtype={"FIPS": "string"}
)

print("Dataset shape:", modeling_data.shape)
print("Unique FIPS:", modeling_data["FIPS"].nunique())
print("Missing target values:", modeling_data["OBESITY_AdjPrev"].isna().sum())

Dataset shape: (3135, 22)
Unique FIPS: 3135
Missing target values: 0


In [4]:
selected_predictors = [
    "PCT_LACCESS_POP19",
    "PCT_LACCESS_LOWI19",
    "GROCPTH20",
    "CONVSPTH20",
    "FFRPTH20",
    "FSRPTH20",
    "MEDHHINC21",
    "POVRATE21",
    "CHILDPOVRATE21",
    "DEEPPOVRATE21",
    "PC_SNAPBEN22",
    "PCT_65OLDER20",
    "PCT_18YOUNGER20",
    "PCT_NHWHITE20",
    "PCT_NHBLACK20",
    "PCT_HISP20",
    "PCT_NHASIAN20",
    "RECFACPTH20"
]

X = modeling_data[selected_predictors].copy()
y = modeling_data["OBESITY_AdjPrev"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Predictors:", len(selected_predictors))

X shape: (3135, 18)
y shape: (3135,)
Predictors: 18


In [5]:
outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Outer CV folds:", outer_cv.get_n_splits())
print("Inner CV folds:", inner_cv.get_n_splits())

Outer CV folds: 5
Inner CV folds: 3


In [6]:
class ThesisPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, missing_threshold=20.0, correlation_threshold=0.80):
        self.missing_threshold = missing_threshold
        self.correlation_threshold = correlation_threshold

    def fit(self, X, y=None):
        X = X.copy()

        # 1. Missingness assessment
        self.missing_summary_ = pd.DataFrame({
            "missing_count": X.isna().sum(),
            "missing_pct": X.isna().mean() * 100
        }).round(2)

        self.excluded_missing_ = self.missing_summary_[
            self.missing_summary_["missing_pct"] > self.missing_threshold
        ].index.tolist()

        self.retained_after_missing_ = [
            col for col in X.columns
            if col not in self.excluded_missing_
        ]

        X_retained = X[self.retained_after_missing_].copy()

        # 2. Median imputation
        self.imputer_ = SimpleImputer(strategy="median")
        self.imputer_.fit(X_retained)

        X_imputed = pd.DataFrame(
            self.imputer_.transform(X_retained),
            columns=self.retained_after_missing_,
            index=X.index
        )

        # 3. Pearson correlation analysis
        self.corr_matrix_ = X_imputed.corr(method="pearson")

        high_corr_pairs = []

        columns = self.corr_matrix_.columns

        for i in range(len(columns)):
            for j in range(i + 1, len(columns)):
                r = self.corr_matrix_.iloc[i, j]

                if abs(r) >= self.correlation_threshold:
                    high_corr_pairs.append({
                        "predictor_1": columns[i],
                        "predictor_2": columns[j],
                        "r": r,
                        "abs_r": abs(r)
                    })

        self.high_corr_pairs_ = pd.DataFrame(high_corr_pairs)

        # Predefined correlation-resolution decisions
        self.correlation_excluded_ = []

        if (
            "POVRATE21" in X_imputed.columns
            and "CHILDPOVRATE21" in X_imputed.columns
            and abs(
                self.corr_matrix_.loc[
                    "POVRATE21",
                    "CHILDPOVRATE21"
                ]
            ) >= self.correlation_threshold
        ):
            self.correlation_excluded_.append(
                "CHILDPOVRATE21"
            )

        if (
            "PCT_LACCESS_POP19" in X_imputed.columns
            and "PCT_LACCESS_LOWI19" in X_imputed.columns
            and abs(
                self.corr_matrix_.loc[
                    "PCT_LACCESS_POP19",
                    "PCT_LACCESS_LOWI19"
                ]
            ) >= self.correlation_threshold
        ):
            self.correlation_excluded_.append(
                "PCT_LACCESS_POP19"
            )

        self.retained_predictors_ = [
            col for col in X_imputed.columns
            if col not in self.correlation_excluded_
        ]

        X_final = X_imputed[self.retained_predictors_].copy()

        # 4. IQR review
        # Extreme values are flagged, not automatically removed.
        iqr_summary = []

        for column in X_final.columns:
            values = X_final[column]

            q1 = values.quantile(0.25)
            q3 = values.quantile(0.75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            outlier_mask = (
                (values < lower_bound)
                | (values > upper_bound)
            )

            iqr_summary.append({
                "predictor": column,
                "Q1": q1,
                "Q3": q3,
                "IQR": iqr,
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "outlier_count": int(outlier_mask.sum()),
                "min": values.min(),
                "max": values.max()
            })

        self.iqr_summary_ = pd.DataFrame(iqr_summary)

        return self

    def transform(self, X):
        X = X.copy()

        # Apply only decisions learned during fit()
        X_retained = X[self.retained_after_missing_].copy()

        X_imputed = pd.DataFrame(
            self.imputer_.transform(X_retained),
            columns=self.retained_after_missing_,
            index=X.index
        )

        return X_imputed[self.retained_predictors_].copy()

In [7]:
train_idx, val_idx = next(outer_cv.split(X))

X_train_outer = X.iloc[train_idx].copy()
X_val_outer = X.iloc[val_idx].copy()

preprocessor_test = ThesisPreprocessor(
    missing_threshold=20.0,
    correlation_threshold=0.80
)

preprocessor_test.fit(X_train_outer)

X_train_processed = preprocessor_test.transform(X_train_outer)
X_val_processed = preprocessor_test.transform(X_val_outer)

print("Training shape before:", X_train_outer.shape)
print("Validation shape before:", X_val_outer.shape)

print("Training shape after:", X_train_processed.shape)
print("Validation shape after:", X_val_processed.shape)

print("\nExcluded for >20% missingness:")
print(preprocessor_test.excluded_missing_)

print("\nExcluded for correlation:")
print(preprocessor_test.correlation_excluded_)

print("\nRetained predictors:")
print(preprocessor_test.retained_predictors_)

print("\nMissing values after preprocessing:")
print("Train:", X_train_processed.isna().sum().sum())
print("Validation:", X_val_processed.isna().sum().sum())

Training shape before: (2508, 18)
Validation shape before: (627, 18)
Training shape after: (2508, 14)
Validation shape after: (627, 14)

Excluded for >20% missingness:
['GROCPTH20', 'RECFACPTH20']

Excluded for correlation:
['CHILDPOVRATE21', 'PCT_LACCESS_POP19']

Retained predictors:
['PCT_LACCESS_LOWI19', 'CONVSPTH20', 'FFRPTH20', 'FSRPTH20', 'MEDHHINC21', 'POVRATE21', 'DEEPPOVRATE21', 'PC_SNAPBEN22', 'PCT_65OLDER20', 'PCT_18YOUNGER20', 'PCT_NHWHITE20', 'PCT_NHBLACK20', 'PCT_HISP20', 'PCT_NHASIAN20']

Missing values after preprocessing:
Train: 0
Validation: 0


In [8]:
rf_pipeline = Pipeline([
    (
        "preprocessor",
        ThesisPreprocessor(
            missing_threshold=20.0,
            correlation_threshold=0.80
        )
    ),
    (
        "rf",
        RandomForestRegressor(
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

print(rf_pipeline)

Pipeline(steps=[('preprocessor', ThesisPreprocessor()),
                ('rf', RandomForestRegressor(n_jobs=-1, random_state=42))])


In [9]:
rf_param_distributions = {
    "rf__n_estimators": [100, 200, 300, 400, 500],
    "rf__max_depth": [None, 5, 10, 15, 20, 30],
    "rf__min_samples_split": [2, 5, 10]
}

print("RF hyperparameters to tune:")

for parameter, values in rf_param_distributions.items():
    print(parameter, ":", values)

RF hyperparameters to tune:
rf__n_estimators : [100, 200, 300, 400, 500]
rf__max_depth : [None, 5, 10, 15, 20, 30]
rf__min_samples_split : [2, 5, 10]


In [10]:
# Get Outer Fold 1
train_idx, val_idx = next(outer_cv.split(X))

X_train_outer = X.iloc[train_idx].copy()
X_val_outer = X.iloc[val_idx].copy()

y_train_outer = y.iloc[train_idx].copy()
y_val_outer = y.iloc[val_idx].copy()

print("Outer Fold 1")
print("Training samples:", len(X_train_outer))
print("Validation samples:", len(X_val_outer))

Outer Fold 1
Training samples: 2508
Validation samples: 627


In [11]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_distributions,
    n_iter=10,
    scoring="neg_root_mean_squared_error",
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train_outer, y_train_outer)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'rf__max_depth': [None, 5, ...], 'rf__min_samples_split': [2, 5, ...], 'rf__n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be 

In [12]:
print("Best RF parameters:")
print(rf_search.best_params_)

print("\nBest inner CV RMSE:")
print(-rf_search.best_score_)

Best RF parameters:
{'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__max_depth': None}

Best inner CV RMSE:
2.8515014563291814


In [13]:
best_rf_model = rf_search.best_estimator_

y_pred_outer = best_rf_model.predict(X_val_outer)

outer_mae = mean_absolute_error(
    y_val_outer,
    y_pred_outer
)

outer_rmse = np.sqrt(
    mean_squared_error(
        y_val_outer,
        y_pred_outer
    )
)

outer_r2 = r2_score(
    y_val_outer,
    y_pred_outer
)

print("Outer Fold 1 Results")
print(f"MAE:  {outer_mae:.4f}")
print(f"RMSE: {outer_rmse:.4f}")
print(f"R²:   {outer_r2:.4f}")

Outer Fold 1 Results
MAE:  2.2954
RMSE: 2.9629
R²:   0.6055


In [14]:
rf_fold_results = []
rf_predictions = []
rf_best_params = []

for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X), start=1):

    print(f"\n{'=' * 50}")
    print(f"OUTER FOLD {fold}")
    print(f"{'=' * 50}")

    # Outer training and validation data
    X_train_outer = X.iloc[train_idx].copy()
    X_val_outer = X.iloc[val_idx].copy()

    y_train_outer = y.iloc[train_idx].copy()
    y_val_outer = y.iloc[val_idx].copy()

    # New pipeline for this outer fold
    rf_pipeline_fold = Pipeline([
        (
            "preprocessor",
            ThesisPreprocessor(
                missing_threshold=20.0,
                correlation_threshold=0.80
            )
        ),
        (
            "rf",
            RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ])

    # Inner 3-fold hyperparameter tuning
    rf_search_fold = RandomizedSearchCV(
        estimator=rf_pipeline_fold,
        param_distributions=rf_param_distributions,
        n_iter=10,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True
    )

    rf_search_fold.fit(
        X_train_outer,
        y_train_outer
    )

    # Best model has been refitted on full outer training data
    best_model = rf_search_fold.best_estimator_

    # Predict untouched outer validation fold
    y_pred = best_model.predict(X_val_outer)

    # Outer-fold evaluation
    mae = mean_absolute_error(
        y_val_outer,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val_outer,
            y_pred
        )
    )

    r2 = r2_score(
        y_val_outer,
        y_pred
    )

    # Save fold metrics
    rf_fold_results.append({
        "fold": fold,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    # Save best hyperparameters
    rf_best_params.append({
        "fold": fold,
        "n_estimators":
            rf_search_fold.best_params_["rf__n_estimators"],
        "max_depth":
            rf_search_fold.best_params_["rf__max_depth"],
        "min_samples_split":
            rf_search_fold.best_params_["rf__min_samples_split"],
        "inner_CV_RMSE":
            -rf_search_fold.best_score_
    })

    # Save held-out predictions
    for idx, actual, predicted in zip(
        val_idx,
        y_val_outer,
        y_pred
    ):
        rf_predictions.append({
            "FIPS": modeling_data.iloc[idx]["FIPS"],
            "fold": fold,
            "actual": actual,
            "predicted": predicted
        })

    print("Best parameters:", rf_search_fold.best_params_)
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}")


OUTER FOLD 1
Best parameters: {'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__max_depth': None}
MAE:  2.2954
RMSE: 2.9629
R²:   0.6055

OUTER FOLD 2
Best parameters: {'rf__n_estimators': 100, 'rf__min_samples_split': 5, 'rf__max_depth': 20}
MAE:  2.3576
RMSE: 3.0443
R²:   0.6032

OUTER FOLD 3
Best parameters: {'rf__n_estimators': 100, 'rf__min_samples_split': 5, 'rf__max_depth': 20}
MAE:  2.2401
RMSE: 2.9819
R²:   0.5948

OUTER FOLD 4
Best parameters: {'rf__n_estimators': 100, 'rf__min_samples_split': 5, 'rf__max_depth': 20}
MAE:  2.0736
RMSE: 2.6468
R²:   0.6608

OUTER FOLD 5
Best parameters: {'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__max_depth': None}
MAE:  2.1486
RMSE: 2.7801
R²:   0.5846


In [15]:
rf_metrics_df = pd.DataFrame(rf_fold_results)

print("Random Forest Outer-Fold Results")
print(rf_metrics_df.round(4))

print("\nRandom Forest Performance Summary")

for metric in ["MAE", "RMSE", "R2"]:
    mean_value = rf_metrics_df[metric].mean()
    sd_value = rf_metrics_df[metric].std()

    print(
        f"{metric}: "
        f"{mean_value:.4f} ± {sd_value:.4f}"
    )

Random Forest Outer-Fold Results
   fold     MAE    RMSE      R2
0     1  2.2954  2.9629  0.6055
1     2  2.3576  3.0443  0.6032
2     3  2.2401  2.9819  0.5948
3     4  2.0736  2.6468  0.6608
4     5  2.1486  2.7801  0.5846

Random Forest Performance Summary
MAE: 2.2231 ± 0.1135
RMSE: 2.8832 ± 0.1648
R2: 0.6098 ± 0.0297


In [16]:
rf_predictions_df = pd.DataFrame(rf_predictions)

print("Prediction rows:", len(rf_predictions_df))
print("Unique FIPS:", rf_predictions_df["FIPS"].nunique())
print("Duplicate FIPS:", rf_predictions_df["FIPS"].duplicated().sum())

print(
    "Missing actual values:",
    rf_predictions_df["actual"].isna().sum()
)

print(
    "Missing predictions:",
    rf_predictions_df["predicted"].isna().sum()
)

print("\nPredictions per fold:")
print(
    rf_predictions_df["fold"]
    .value_counts()
    .sort_index()
)

Prediction rows: 3135
Unique FIPS: 3135
Duplicate FIPS: 0
Missing actual values: 0
Missing predictions: 0

Predictions per fold:
fold
1    627
2    627
3    627
4    627
5    627
Name: count, dtype: int64


In [17]:
rf_summary_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Mean": [
        rf_metrics_df["MAE"].mean(),
        rf_metrics_df["RMSE"].mean(),
        rf_metrics_df["R2"].mean()
    ],
    "SD": [
        rf_metrics_df["MAE"].std(),
        rf_metrics_df["RMSE"].std(),
        rf_metrics_df["R2"].std()
    ]
})

rf_best_params_df = pd.DataFrame(rf_best_params)

print("Performance Summary:")
display(rf_summary_df.round(4))

print("\nBest Hyperparameters by Outer Fold:")
display(rf_best_params_df.round(4))

Performance Summary:


,Metric,Mean,SD
0,MAE,2.2231,0.1135
1,RMSE,2.8832,0.1648
2,R2,0.6098,0.0297



Best Hyperparameters by Outer Fold:


,fold,n_estimators,max_depth,min_samples_split,inner_CV_RMSE
0,1,100,NaN,2,2.8515
1,2,100,20.0,5,2.9176
2,3,100,20.0,5,2.9012
3,4,100,20.0,5,2.9623
4,5,100,NaN,2,2.9410


In [18]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "random_forest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rf_metrics_df.to_csv(
    OUTPUT_DIR / "rf_outer_fold_metrics.csv",
    index=False
)

rf_summary_df.to_csv(
    OUTPUT_DIR / "rf_performance_summary.csv",
    index=False
)

rf_predictions_df.to_csv(
    OUTPUT_DIR / "rf_outer_predictions.csv",
    index=False
)

rf_best_params_df.to_csv(
    OUTPUT_DIR / "rf_best_hyperparameters.csv",
    index=False
)

print("Saved Random Forest outputs to:")
print(OUTPUT_DIR)

print("\nFiles:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)

Saved Random Forest outputs to:
/Users/chelsearose/Documents/GitHub/county-obesity-prediction/outputs/random_forest

Files:
- rf_best_hyperparameters.csv
- rf_outer_fold_metrics.csv
- rf_outer_predictions.csv
- rf_performance_summary.csv
